In [1]:
import os

import numpy as np
import pandas as pd
from scipy.stats import kurtosis

# ------------------------------------------------------------------------------
# 1. LOAD RAW PARQUET DATASET FROM ROOT DATA FOLDER
# ------------------------------------------------------------------------------
input_path = "../data/raw/market_data_raw.parquet"
df = pd.read_parquet(input_path)

print(f"Loaded {len(df):,} records for {df['Ticker'].nunique()} tickers.")


# ------------------------------------------------------------------------------
# 2. DEFINE ROLLING CALCULATION FUNCTIONS
# ------------------------------------------------------------------------------
def compute_rolling_drawdown(series, window=21):
    """
    Calculates rolling peak-to-trough drawdown over a window of trading days (~21 days = 30 calendar days).
    """
    rolling_max = series.rolling(window=window, min_periods=window).max()
    drawdown = (series - rolling_max) / rolling_max
    return drawdown


def compute_rolling_kurtosis(returns, window=63):
    """
    Calculates rolling excess kurtosis over a lookback window (e.g. 63 trading days ~ 3 months).
    """
    return returns.rolling(window=window, min_periods=window).apply(
        lambda x: kurtosis(x, fisher=True), raw=True
    )


# ------------------------------------------------------------------------------
# 3. FEATURE EXTRACTION LOOP BY TICKER
# ------------------------------------------------------------------------------
processed_frames = []

for ticker, group in df.groupby("Ticker"):
    group = group.sort_values("Date").copy()

    # Log returns
    returns = group["Log_Return"].dropna()

    # Feature 1: Rolling 30-Day Annualized Volatility (21 trading days)
    group["Vol_30D_Ann"] = returns.rolling(window=21).std() * np.sqrt(252)

    # Feature 2: Rolling 63-Day Excess Kurtosis (Tail Thickness)
    group["Kurtosis_63D"] = compute_rolling_kurtosis(returns, window=63)

    # Target Variable: 21-Day Rolling Max Drawdown
    group["Max_Drawdown_30D"] = compute_rolling_drawdown(group["Adj_Close"], window=21)

    # Binary Label: Institutional Distress (Drawdown >= 40% -> 1, else 0)
    group["Distress_Label"] = (group["Max_Drawdown_30D"].abs() >= 0.40).astype(int)

    processed_frames.append(group)

master_features = pd.concat(processed_frames, ignore_index=True)

# ------------------------------------------------------------------------------
# 4. SAVE PROCESSED DATASET TO DATA/PROCESSED/
# ------------------------------------------------------------------------------
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_parquet = os.path.join(output_dir, "market_data_features.parquet")
master_features.to_parquet(output_parquet, index=False)
output_parquet = os.path.join(output_dir, "market_data_features.csv")
master_features.to_csv(output_parquet, index=False)

print(f"\n[Saved] Processed dataset with features written to: {output_parquet}")
print(
    f"Total Distress Events Flagged (Label=1): {master_features['Distress_Label'].sum():,}"
)

Loaded 79,046 records for 19 tickers.

[Saved] Processed dataset with features written to: ../data/processed/market_data_features.csv
Total Distress Events Flagged (Label=1): 874


In [14]:
import pandas as pd

# Load your processed feature dataset
df_features = pd.read_parquet("../data/processed/market_data_features.parquet")

# See breakdown of distress event counts by ticker
distress_summary = (
    df_features[df_features["Distress_Label"] == 1]
    .groupby(["Ticker", "Name", "Category"])
    .size()
    .reset_index(name="Distress_Days_Count")
    .sort_values(by="Ticker", ascending=True)
)

print("Distress Events by Institution:\n")
print(distress_summary.to_string(index=False))

Distress Events by Institution:

Ticker                         Name         Category  Distress_Days_Count
   AIG American International Group  Distressed_2008                   96
   BAC              Bank of America Anchor_Financial                   48
     C                    Citigroup  Distressed_2008                   66
  FITB          Fifth Third Bancorp    Regional_Bank                   60
  FRCB          First Republic Bank  Distressed_2023                  161
    GS                Goldman Sachs Anchor_Financial                   11
   HTZ        Hertz Global Holdings  Distressed_2020                    1
   IVR     Invesco Mortgage Capital  Distressed_2020                   41
   JPM               JPMorgan Chase Anchor_Financial                    6
   KEY                      KeyCorp    Regional_Bank                   37
   MFA                MFA Financial  Distressed_2020                   28
    MS               Morgan Stanley Anchor_Financial                   29
   TW

In [ ]:
import pandas as pd

# Load raw data
df_raw = pd.read_parquet("../data/raw/market_data_raw.parquet")

# Check date ranges and record counts for Lehman and SVB
check_tickers = ["LEHMQ", "SIVB", "BSC"]
subset = df_raw[df_raw["Ticker"].isin(check_tickers)]

print(subset.groupby("Ticker")["Date"].agg(["min", "max", "count"]))

Empty DataFrame
Columns: [min, max, count]
Index: []


In [ ]:
import pandas as pd

# Load master feature dataset
df_features = pd.read_parquet("../data/processed/market_data_features.parquet")

# Check available tickers and their date spans
ticker_audit = (
    df_features.groupby("Ticker")["Date"].agg(["min", "max", "count"]).reset_index()
)
print("Ticker Audit:\n")

print(ticker_audit.to_string(index=False))

Ticker Audit:

Ticker        min        max  count
   AIG 2006-01-03 2023-12-29   4529
   BAC 2006-01-03 2023-12-29   4529
     C 2006-01-03 2023-12-29   4529
  FITB 2006-01-03 2023-12-29   4529
  FRCB 2010-12-09 2023-12-29   3286
    GS 2006-01-03 2023-12-29   4529
   HTZ 2021-07-01 2023-12-29    629
   IVR 2009-07-01 2023-12-29   3650
   JPM 2006-01-03 2023-12-29   4529
   KEY 2006-01-03 2023-12-29   4529
   MFA 2006-01-03 2023-12-29   4529
    MS 2006-01-03 2023-12-29   4529
   TWO 2009-10-30 2023-12-29   3565
   WAL 2006-01-03 2023-12-29   4529
   XLE 2006-01-03 2023-12-29   4529
   XLF 2006-01-03 2023-12-29   4529
   XLK 2006-01-03 2023-12-29   4529
 ^GSPC 2006-01-03 2023-12-29   4529
  ^VIX 2006-01-03 2023-12-29   4529


In [ ]:
# %% [code]
import os

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# 1. LOAD RAW DATASET & INTEGRATE PATCH TICKERS
# ------------------------------------------------------------------------------
input_path = "../data/raw/market_data_raw.parquet"
df = pd.read_parquet(input_path)

# Define patch files to include (excluding SBNY and CHK as requested)
patch_files = [
    "../data/raw/patches/BSC.csv",
    "../data/raw/patches/LEHMQ.csv",
    "../data/raw/patches/PACW.csv",
    "../data/raw/patches/SIVB.csv",
]

patch_frames = []
for p_file in patch_files:
    if os.path.exists(p_file):
        p_df = pd.read_csv(p_file)
        # Ensure standard schema (Date, Adj_Close, Log_Return, Ticker, Name, Category, etc.)
        # Standardize column names if necessary
        if "Ticker" not in p_df.columns:
            ticker_name = os.path.basename(p_file).split(".")[0]
            p_df["Ticker"] = ticker_name
        patch_frames.append(p_df)

if patch_frames:
    df_patches = pd.concat(patch_frames, ignore_index=True)
    # Ensure date alignment
    df_patches["Date"] = pd.to_datetime(df_patches["Date"])
    df["Date"] = pd.to_datetime(df["Date"])

    # Combine main raw data with patches, dropping duplicates if any
    df = pd.concat([df, df_patches], ignore_index=True).drop_duplicates(
        subset=["Date", "Ticker"], keep="last"
    )

print(
    f"Loaded {len(df):,} total records for {df['Ticker'].nunique()} unique"
    f" tickers (including patches)."
)


# ------------------------------------------------------------------------------
# 2. DEFINE ROLLING CALCULATION FUNCTIONS
# ------------------------------------------------------------------------------
def compute_rolling_drawdown(series, window=21):
    """Calculates rolling peak-to-trough drawdown over a window of trading days (~21 days = 30 calendar days)."""
    rolling_max = series.rolling(window=window, min_periods=window).max()
    drawdown = (series - rolling_max) / rolling_max
    return drawdown


def compute_rolling_kurtosis(returns, window=63):
    """Calculates rolling excess kurtosis over a lookback window (e.g. 63 trading days ~ 3 months)."""
    return returns.rolling(window=window, min_periods=window).apply(
        lambda x: kurtosis(x, fisher=True), raw=True
    )


# ------------------------------------------------------------------------------
# 3. FEATURE EXTRACTION LOOP BY TICKER
# ------------------------------------------------------------------------------
processed_frames = []

for ticker, group in df.groupby("Ticker"):
    group = group.sort_values("Date").copy()

    # Log returns (ensure column name consistency)
    if "Log_Return" not in group.columns and "Adj_Close" in group.columns:
        group["Log_Return"] = np.log(group["Adj_Close"] / group["Adj_Close"].shift(1))

    returns = group["Log_Return"].dropna()

    # Feature 1: Rolling 30-Day Annualized Volatility (21 trading days)
    group["Vol_30D_Ann"] = returns.rolling(window=21).std() * np.sqrt(252)

    # Feature 2: Rolling 63-Day Excess Kurtosis (Tail Thickness)
    group["Kurtosis_63D"] = compute_rolling_kurtosis(returns, window=63)

    # Target Variable: 21-Day Rolling Max Drawdown
    group["Max_Drawdown_30D"] = compute_rolling_drawdown(group["Adj_Close"], window=21)

    # Binary Label: Institutional Distress (Drawdown >= 40% -> 1, else 0)
    group["Distress_Label"] = (group["Max_Drawdown_30D"].abs() >= 0.40).astype(int)

    processed_frames.append(group)

master_features = pd.concat(processed_frames, ignore_index=True)

# ------------------------------------------------------------------------------
# 4. SAVE PROCESSED DATASET TO DATA/PROCESSED/
# ------------------------------------------------------------------------------
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_parquet = os.path.join(output_dir, "market_data_features.parquet")
master_features.to_parquet(output_parquet, index=False)
print(f"\n[Saved] Processed dataset with features written to: {output_csv}")
print(
    f"Total Distress Events Flagged (Label=1):"
    f" {master_features['Distress_Label'].sum():,}"

output_csv = os.path.join(output_dir, "market_data_features.csv")
master_features.to_csv(output_csv, index=False)
print(f"\n[Saved] Processed dataset with features written to: {output_csv}")
print(
    f"Total Distress Events Flagged (Label=1):"
    f" {master_features['Distress_Label'].sum():,}"
)

Loaded 79,046 total records for 19 unique tickers (including patches).

[Saved] Processed dataset with features written to: ../data/processed/market_data_features.csv
Total Distress Events Flagged (Label=1): 874
